In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/shashwatkumarjha/trainedweights/best.pt
/kaggle/input/datasets/shashwatkumarjha/chotudon/ChotiClip.mp4


In [2]:
!pip install ultralytics
!pip install boxmot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 41.1 MB/s eta 0:00:00
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=3a90f58d57524ef85794cd4b5222745ad6ec6a48845422f83cda95b94c59ed6d
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy
  Attempting uninstall: regex
    Found 

In [3]:
from ultralytics import YOLO

model = YOLO('/kaggle/input/datasets/shashwatkumarjha/trainedweights/best.pt')   # path to your weights

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
import cv2

cap = cv2.VideoCapture('/kaggle/input/datasets/shashwatkumarjha/chotudon/ChotiClip.mp4')

In [5]:
roi_points = [[540, 1066],[972, 216],     
    [1080, 216],   
    [1650, 1066]]
pts = np.array(roi_points, np.int32)

In [6]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, 10.0, (320,240))  # 🔥 smaller size

In [7]:
import cv2
import numpy as np
from ultralytics import YOLO

# LOAD MODEL
model = YOLO('/kaggle/input/datasets/shashwatkumarjha/trainedweights/best.pt')
model.to('cpu')

# VIDEO
cap = cv2.VideoCapture('/kaggle/input/datasets/shashwatkumarjha/chotudon/ChotiClip.mp4')

# GET ORIGINAL SIZE
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("Video size:", width, height)

# VIDEO WRITER
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('outputnow.mp4', fourcc, 10.0, (width, height))

# YOUR ROI (UNCHANGED)
roi_points = [[540,1066],[972,216],[1080,216],[1650,1066]]
pts = np.array(roi_points, np.int32)

frame_written = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # MASK
    mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 255)

    masked = cv2.bitwise_and(frame, frame, mask=mask)

    # YOLO
    results = model(masked, verbose=False)

    annotated = results[0].plot()

    # DRAW ROI (debug)
    cv2.polylines(annotated, [pts], True, (0,255,0), 3)

    out.write(annotated)
    frame_written += 1

cap.release()
out.release()

print("Frames written:", frame_written)

Video size: 1920 1080
Frames written: 3194


In [8]:
# ==============================
# 1. IMPORTS
# ==============================
import cv2
import numpy as np
from ultralytics import YOLO

# ==============================
# 2. LOAD MODEL
# ==============================
model = YOLO('/kaggle/input/datasets/shashwatkumarjha/trainedweights/best.pt')
model.to('cpu')   # safer

# ==============================
# 3. LOAD VIDEO
# ==============================
cap = cv2.VideoCapture('/kaggle/input/datasets/shashwatkumarjha/chotudon/ChotiClip.mp4')

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# ==============================
# 4. VIDEO WRITER
# ==============================
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('tracked_output.mp4', fourcc, 10.0, (width, height))

# ==============================
# 5. ROI (YOUR POINTS)
# ==============================
roi_points = [[540,1066],[972,216],[1080,216],[1650,1066]]
pts = np.array(roi_points, np.int32)

# ==============================
# 6. TRACK HISTORY
# ==============================
track_history = {}

# ==============================
# 7. PROCESS LOOP
# ==============================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # --------------------------
    # APPLY MASK
    # --------------------------
    mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 255)
    masked = cv2.bitwise_and(frame, frame, mask=mask)

    # --------------------------
    # YOLO TRACKING
    # --------------------------
    results = model.track(masked, persist=True, verbose=False)

    annotated = results[0].plot()

    # --------------------------
    # TRAJECTORY LOGIC
    # --------------------------
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu()
        ids = results[0].boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, ids):
            x, y, w, h = box
            center = (int(x), int(y))

            if track_id not in track_history:
                track_history[track_id] = []

            track_history[track_id].append(center)

            # keep last 30 points
            if len(track_history[track_id]) > 30:
                track_history[track_id].pop(0)

    # --------------------------
    # DRAW TRAJECTORIES
    # --------------------------
    for track_id, points in track_history.items():
        for i in range(1, len(points)):
            cv2.line(annotated, points[i-1], points[i], (0,0,255), 2)

    # --------------------------
    # DRAW ROI BORDER
    # --------------------------
    cv2.polylines(annotated, [pts], True, (0,255,0), 3)

    # --------------------------
    # SAVE FRAME
    # --------------------------
    out.write(annotated)

# ==============================
# 8. CLEANUP
# ==============================
cap.release()
out.release()

print("✅ Tracking video saved as tracked_output.mp4")

✅ Tracking video saved as tracked_output.mp4
